# RankResume — Document to PostgreSQL Pipeline

This notebook performs local document extraction, technology-safe NLP cleaning, structured JSON extraction, duplicate detection, and PostgreSQL JSONB storage. It does not implement matching, ATS scoring, ranking, testing, or behaviour analysis.

## 1. Install dependencies if needed

In [1]:
# Run once in the notebook kernel if imports are unavailable.
# %pip install pdfplumber python-docx psycopg2-binary

## 2. Import the modular pipeline

In [2]:
import sys
from pathlib import Path

DB_STORE = Path('/home/vedant-2506/Desktop/Python_ML/Project/DB_Store')
if str(DB_STORE) not in sys.path:
    sys.path.insert(0, str(DB_STORE))

from rankresume_pipeline import *

print('Configured resumes path:', RESUMES_DIR)
print('Configured JDs path:', JDS_DIR)
print('Supported extensions:', sorted(SUPPORTED_EXTENSIONS))

Configured resumes path: /home/vedant-2506/Desktop/Python_ML/Project/DB_Store/Data/resumes
Configured JDs path: /home/vedant-2506/Desktop/Python_ML/Project/DB_Store/Data/jds
Supported extensions: ['.doc', '.docx', '.pdf']


## 3. Verify input folders and local readers

In [3]:
resume_dir, jd_dir = resolve_input_dirs()
print('Using resumes:', resume_dir)
print('Using JDs:', jd_dir)
print('Resume files:', len(files_in(resume_dir)))
print('JD files:', len(files_in(jd_dir)))
print('pdfplumber reader: available')
print('python-docx reader: available')
print('DOC reader: antiword or LibreOffice is checked only when a .doc file is encountered')

Using resumes: /home/vedant-2506/Desktop/Python_ML/Project/Data/resumes
Using JDs: /home/vedant-2506/Desktop/Python_ML/Project/Data/jds
Resume files: 19
JD files: 1
pdfplumber reader: available
python-docx reader: available
DOC reader: antiword or LibreOffice is checked only when a .doc file is encountered


## 4. Test extraction, contacts, cleaning, and technology protection

In [4]:
sample = 'C++ C# .NET ASP.NET Node.js React.js Next.js CI/CD Spring Boot REST API PostgreSQL Machine Learning JavaScript'
print('Detected technologies:', technology_matches(sample))
print('Cleaned sample:', clean_text(sample))

supported_samples = [path for path in files_in(resume_dir) if path.suffix.lower() in SUPPORTED_EXTENSIONS][:3]
for sample_path in supported_samples:
    try:
        text = extract_text(sample_path)
        print(f'\n{sample_path.name}: {len(text)} extracted characters')
        print(extract_contacts(text))
        print(parse_resume(text))
    except Exception as error:
        print(f'{sample_path.name}: {type(error).__name__}: {error}')

Detected technologies: ['C++', 'C#', '.NET', 'Node.js', 'React.js', 'Next.js', 'CI/CD', 'Spring Boot', 'REST API', 'PostgreSQL', 'Machine Learning', 'JavaScript']
Cleaned sample: C++ C# .NET .NET Node.js React.js Next.js CI/CD Spring Boot REST API PostgreSQL Machine Learning JavaScript

Aarya_Kale.pdf: 4281 extracted characters
{'name': 'AARYA KALE', 'email': 'kaleaarya704@gmail.com', 'phone': '8421749236'}
{'skill': ['HTML5', 'CSS3', 'React.js', 'REST API', 'Java', 'C++', 'Spring Boot', 'MySQL', 'Git', 'GitHub'], 'education': [{'degree': 'M.Sc. in Scientific Computing | Savitribai Phule Pune University | Aug 2025 - May 2026', 'mark': ''}, {'degree': 'B.Sc. in Computer Science | Savitribai Phule Pune University | 2021 - 2024 | CGPA: 7.69', 'mark': 'CGPA: 7.69'}, {'degree': 'HSC (12th Grade) - General Science | Maharashtra State Board | 2021 | 85.83%', 'mark': '85.83%'}, {'degree': 'SSC (10th Grade) | Maharashtra State Board | 2019 | 87.80%', 'mark': '87.80%'}], 'project': [{'project_ti

## 5. PostgreSQL safety check and schema setup

`RESET_TABLES` is deliberately false. Set it to true only after reviewing the public-table list printed by this cell. The reset function connects to `rankresume`, verifies the database, inspects tables, and drops only the three RankResume tables.

In [5]:
RESET_TABLES = False
connection = connect_db()
print('Verified database:', DB_NAME)
print('Existing public tables:', inspect_tables(connection))
create_schema(connection, reset=RESET_TABLES)
seed_dictionary(connection)
print('Schema and technology dictionary are ready.')

Verified database: rankresume
Existing public tables: ['candidate', 'job_description', 'technology_dictionary']
Schema and technology dictionary are ready.


## 6. Run the complete batch

Each document is handled independently. A failed or unsupported file is logged and does not stop the remaining batch.

In [6]:
summary = process_batch(connection, reset=False)
print_summary(summary)

INFO: Duplicate resume skipped: Aarya_Kale.pdf
INFO: Duplicate resume skipped: Abhijeet.pdf
INFO: Duplicate resume skipped: Adity_Gowari.pdf
INFO: Duplicate resume skipped: Aishwarya_Khedkar.pdf
INFO: Duplicate resume skipped: Aniket_Jadhav_Resume.pdf
INFO: Duplicate resume skipped: Arya_Gavali_resume (2).pdf
INFO: Duplicate resume skipped: Khumba_Lunganlung.pdf
INFO: Duplicate resume skipped: Kunal_Kamod.pdf
INFO: Duplicate resume skipped: Pankaj_Jadhav.pdf
INFO: Duplicate resume skipped: Prapti_Gagare.pdf
INFO: Duplicate resume skipped: Pratiksha_Purkar.pdf
INFO: Duplicate resume skipped: Saily_Kadam.pdf
INFO: Duplicate resume skipped: Sanika_Bhade.pdf
INFO: Duplicate resume skipped: Sanika_Mahind.pdf
INFO: Duplicate resume skipped: Shravani_Pilane.pdf
INFO: Duplicate resume skipped: Siddhesh_Jadhav.pdf
INFO: Duplicate resume skipped: Siddhi_Lagad.pdf
INFO: Duplicate resume skipped: Vedant_Divate.pdf
INFO: Duplicate resume skipped: shraddha_chavan .pdf
INFO: JD already stored; skippe

RankResume DB Store Summary

Resume files found: 19
Successfully processed: 0
Duplicates skipped: 19
Failed: 0
Unsupported: 0

JD files found: 1
Successfully processed: 0
Failed: 0
Unsupported: 0

Database: rankresume
Candidate records: 19
Job Description records: 1



## 7. Validate stored values

In [7]:
with connection.cursor() as cursor:
    cursor.execute('''
        SELECT candidate_id, candidate_name, email, phone, resume_file_name,
               length(raw_text) AS raw_length, length(clean_text) AS clean_length,
               jsonb_typeof(structured_data) AS structured_type
        FROM candidate ORDER BY candidate_id
    ''')
    for row in cursor.fetchall():
        print(row)

with connection.cursor() as cursor:
    cursor.execute('''
        SELECT jd_id, job_title, jd_file_name, length(raw_text),
               length(clean_text), jsonb_typeof(structured_data)
        FROM job_description ORDER BY jd_id
    ''')
    for row in cursor.fetchall():
        print(row)

(1, 'AARYA KALE', 'kaleaarya704@gmail.com', '8421749236', 'Aarya_Kale.pdf', 4281, 3816, 'object')
(2, 'Abhijeet Wargad', 'wargadabhi@gmail.com', '7887497782', 'Abhijeet.pdf', 3287, 2902, 'object')
(3, 'Aditya Gowari', 'adityagowari0722@gmail.com', '+91 77678 30786', 'Adity_Gowari.pdf', 4158, 3653, 'object')
(4, 'Aishwarya Khedkar', 'aishwaryank2025@gmail.com', '+919890304043', 'Aishwarya_Khedkar.pdf', 1857, 1749, 'object')
(5, 'Aniket Sandip Jadhav', 'aniketsjadhav567@gmail.com', '9309108587', 'Aniket_Jadhav_Resume.pdf', 1783, 1616, 'object')
(6, 'Arya Gavali', 'aryapgavali@gmail.com', None, 'Arya_Gavali_resume (2).pdf', 3894, 3418, 'object')
(7, 'Khumba Lunganlung', 'lunganlungkhumba2003@gmail.com', None, 'Khumba_Lunganlung.pdf', 2416, 2184, 'object')
(8, 'KUNAL KAMOD', 'kunalkamod03@gmail.com', '7767859953', 'Kunal_Kamod.pdf', 4753, 4154, 'object')
(9, 'Pankaj Rajendra Jadhav', 'pankajjadhav1712@gmail.com', '+91 9673919286', 'Pankaj_Jadhav.pdf', 2654, 2321, 'object')
(10, 'Prapti Gag

In [8]:
connection.close()